# NB00 - Localiser et analyser les jeux de données avec Unity Catalog

## Mise en place

Exécuter la commande suivante pour créer les :
- Catalogue
- Schéma
- Volumes
- Tables
nécéssaires pour suivre le notebook.

In [0]:
%run "./Resources/NB00/Setup"

## Exploration avec Unity Catalog

Unity Catalog est la solution de gouvernance des données d'Databricks. Il permet d'explorer, gérer et sécuriser tous les actifs de données (tables, vues, volumes, modèles, etc.) dans un espace centralisé.

### 1. Explorer le catalogue avec l'explorateur de données (Catalog Explorer)

Le **Catalog Explorer** est accessible depuis la barre latérale de Databricks. Il permet de :

- Naviguer dans la hiérarchie : **Catalogue → Schéma (base de données) → Table / Vue / Volume**
- Visualiser le schéma des tables, les types de colonnes et des statistiques
- Consulter les métadonnées (description, propriétaire, tags)
- Voir un aperçu des données (échantillon de lignes)
- Gérer les permissions directement depuis l'interface

### 2. Explorer par SQL

On peut interroger les métadonnées d'Unity Catalog directement en SQL :

```sql
-- Lister les catalogues disponibles
SHOW CATALOGS;

-- Lister les schémas d'un catalogue
SHOW SCHEMAS IN my_catalog;

-- Lister les tables d'un schéma
SHOW TABLES IN my_catalog.my_schema;

-- Décrire une table (colonnes, types, commentaires)
DESCRIBE TABLE my_catalog.my_schema.my_table;

-- Décrire en détail (avec statistiques)
DESCRIBE DETAIL my_catalog.my_schema.my_table;

-- Voir le tableau de données
SELECT * FROM my_catalog.my_schema.my_table LIMIT 10;
```

### 3. Explorer par Python (PySpark)

On peut aussi explorer les actifs de données en code Python :

```python
# Lister les catalogues
display(spark.sql("SHOW CATALOGS"))

# Lister les tables d'un schéma
spark.catalog.listTables("my_catalog.my_schema")

# Lire une table dans un DataFrame
spark.read.table("my_catalog.my_schema.my_table")
```

### 4. Volumes

Les **Volumes** d'Unity Catalog servent à stocker des fichiers non tabulaires (CSV, JSON, images, etc.) dans un schéma. Ils permettent d'appliquer les mêmes contrôles de sécurité que pour les tables.

```sql
-- Lister les volumes
SHOW VOLUMES IN my_catalog.my_schema;

-- Lire un fichier depuis un volume
SELECT * FROM read_csv('/Volumes/my_catalog/my_schema/my_volume/data.csv')
```

### 5. Gouvernance et sécurité

Unity Catalog unifie la gouvernance des données :

- **Permissions granulaires** : `GRANT`, `REVOKE` au niveau catalogue, schéma, table, colonne
- **Tags** : étiquetage des actifs pour la classification des données
- **Data Lineage** : traçabilité automatique des flux de données entre les actifs
- **Audit logs** : journalisation des accès aux données via les *system tables*
- **Row-level & column-level security** : filtres de lignes et masques de colonnes

### Schéma de la hiérarchie

```
Metastore (niveau compte)
└── Catalog (ex : my_catalog)
    └── Schema / Database (ex : my_schema)
        ├── Tables
        ├── Views
        └── Volumes
```

> 💡 **Bonne pratique** : Toujours utiliser des noms complets en trois parties (`catalog.schema.table`) pour garantir l'unicité et la clarté des références.



## Démo

Dans la première partie du Notebook, la cellule de mise en place a créée :
- 1 Catalogue
- 1 schéma
- 1 volumes (avec 6 fichiers)
- 6 tables

Pour explorer ces différents éléments il faut cliquer soit sur l'onglet **Catalog** pour avoir la vue catalogue complète ou cliquer sur l'icone à côter de l'explorateur de l'espace de travail pour avoir la vue **Catalog** à côté du Notebook : 

![image_1790775066505.png](./Resources/image_1790775066505.png "image_1790775066505.png")

### 1. Le Catalogue

Un **catalogue** dans Databricks est le conteneur de plus haut niveau dans la hiérarchie d'Unity Catalog. Il regroupe un ou plusieurs **schémas** (bases de données), qui contiennent eux-mêmes des **tables**, des **vues** et des **volumes**.

Chaque catalogue permet de :

- **Organiser** les actifs de données par domaine, projet ou équipe
- **Gérer les accès** de manière centralisée : les permissions accordées sur un catalogue se propagent à tous ses schémas et tables (modèle d'héritage)
- **Isoler** les données entre différents environnements (ex. un catalogue `prod`, un catalogue `dev`)
- **Appliquer une gouvernance unifiée** : tags, lineage, audit, sécurité au niveau ligne et colonne

> 💡 Dans le référencement des données, on utilise toujours le nom complet en trois parties : `catalogue.schéma.table`.

Dans notre cas : `labuser_<username>`

![image_1790775270918.png](./Resources/image_1790775270918.png "image_1790775270918.png")

On accès aux différents éléments du Catalogue: 
- Vue d'ensemble (avec les différents schémas disponibles)
- Détails du catalogue

![image_1790775838591.png](./Resources/image_1790775838591.png "image_1790775838591.png")

- Les politiques (sécurités, accès, ...)
- Les autorisations
- Les workspaces autorisées


### 2. Les Schémas

Un **schéma** (aussi appelé *base de données*) est le deuxième niveau de la hiérarchie d'Unity Catalog. Il se trouve à l'intérieur d'un **catalogue** et contient des tables, des vues et des volumes.

Chaque schéma permet de :

- **Regrouper** logiquement des tables et volumes d'un même domaine ou projet
- **Gérer les accès** au niveau du schéma : les permissions accordées sur un schéma se propagent à toutes ses tables et vues (héritage)
- **Organiser** les données de façon cohérente au sein d'un catalogue

> 💡 Le nom complet d'une table utilise toujours la notation en trois parties : `catalogue.schéma.table`.

Dans notre cas, le schéma s'appelle `sql_analytics` :

On accède aux différents éléments du schéma :
- Vue d'ensemble (avec les tables, vues et volumes disponibles)
- Détails du schéma
- Les autorisations

![image_1790776498118.png](./Resources/image_1790776498118.png "image_1790776498118.png")


### 3. Les Tables

Une **table** est le troisième niveau de la hiérarchie d'Unity Catalog. Elle se trouve à l'intérieur d'un **schéma** et contient les données structurées (lignes et colonnes).

Databricks supporte plusieurs types de tables :

- **Managed table** (table gérée) : Databricks gère à la fois les métadonnées et les fichiers de données
- **External table** (table externe) : les données sont stockées dans un emplacement externe, Databricks ne gère que les métadonnées
- **Delta table** : format par défaut, avec support des transactions ACID, du time travel et du streaming

Chaque table permet de :

- **Stocker** des données structurées interrogables en SQL ou Python
- **Visualiser** le schéma (colonnes, types, commentaires) et un aperçu des données
- **Gérer les accès** au niveau de la table, de la colonne ou de la ligne
- **Suivre le lineage** : voir les tables sources et les dépendances descendantes

> 💡 On accède aux différents éléments d'une table depuis le Catalog Explorer :
> - Vue d'ensemble (schéma des colonnes, échantillon de données)
>   - On peut appliquer 
> - Détails (type de table, emplacement, propriétaire)
> - Lineage (graphique des dépendances)
> - Autorisations
> - Les données (aperçu des lignes)


![image_1790776822413.png](./Resources/image_1790776822413.png "image_1790776822413.png")



![image_1790776693004.png](./Resources/image_1790776693004.png "image_1790776693004.png")


### 4. Marketplace

Le **Databricks Marketplace** est une plateforme d'échange de données qui permet de découvrir, acheter et vendre des jeux de données, des modèles d'IA et des actifs analytiques entre organisations.

Il fonctionne avec **Delta Sharing**, le protocole ouvert de Databricks pour le partage sécurisé de données, sans besoin de les copier.

Chaque offre du Marketplace permet de :

- **Découvrir** des datasets tiers (financiers, météorologiques, marketing, etc.) et des modèles pré-entraînés
- **Accéder** instantanément aux données partagées directement depuis son workspace Databricks
- **Gérer les accès** de manière centralisée grâce à Unity Catalog
- **Partager** ses propres données en tant que fournisseur (*provider*)

> 💡 Les données du Marketplace sont accessibles en lecture seule via des *listings* que l'on peut ajouter à son catalogue en un clic, puis interroger en SQL ou Python comme n'importe quelle table.


